In [1]:
import os
from dotenv import load_dotenv
from langsmith import Client

load_dotenv()

api_key = os.getenv("langsmith_api_key")
if not api_key:
    raise ValueError("Missing langsmith_api_key in .env file.")

os.environ["LANGSMITH_API_KEY"] = api_key
os.environ["LANGCHAIN_API_KEY"] = api_key
os.environ["OPENAI_API_KEY"] = os.getenv("openai", "")
os.environ["LANGCHAIN_TRACING_V2"] = "false"

In [2]:
client = Client()

dataset_name = "chatbot evaluation"
dataset = client.create_dataset(dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {"input": "What is the capital of France?", "output": "The capital of France is Paris."},
        {"input": "What is the largest planet in our solar system?", "output": "The largest planet in our solar system is Jupiter."},
        {"input": "Who wrote the play 'Romeo and Juliet'?", "output": "The play 'Romeo and Juliet' was written by William Shakespeare."},
        {"input": "What is the boiling point of water?", "output": "The boiling point of water is 100 degrees Celsius (212 degrees Fahrenheit) at standard atmospheric pressure."},
        {"input": "What is the currency of Japan?", "output": "The currency of Japan is the Japanese Yen (JPY)."},
    ]
)

{'example_ids': ['96a1b3e3-0e2d-4ea9-86fe-be587f6472e8',
  '339e7fce-be8c-4e43-b19f-75750c125efd',
  '58578bd9-d261-450f-9b09-f94e5b2349e8',
  '6e50ab27-e0d6-45c9-8170-64e47be9f45d',
  '85fec5dc-ec78-47be-a85e-ed25c4fda2cc'],
 'count': 5,
 'as_of': '2026-10-03T14:52:37.155971175Z'}

In [3]:
client = Client()

dataset = client.create_dataset(
    dataset_name="ds-notable-nurture-27", description="A sample dataset in LangSmith."
)
examples = [
    {
        "inputs": {"question": "Which country is Mount Kilimanjaro located in?"},
        "outputs": {"answer": "Mount Kilimanjaro is located in Tanzania."},
    },
    {
        "inputs": {"question": "What is Earth's lowest point?"},
        "outputs": {"answer": "Earth's lowest point is The Dead Sea."},
    },
]
client.create_examples(dataset_id=dataset.id, examples=examples)

{'example_ids': ['13d86594-a340-485d-8dd7-6a13968ecdfc',
  '7b8df908-2acc-4e63-a17c-2a6e04446a3f'],
 'count': 2,
 'as_of': '2026-10-03T14:52:50.163384123Z'}

In [27]:
import openai
from langsmith import wrappers
openai_client = wrappers.wrap_openai(openai.OpenAI()) #path openai client to make it traceble
eval_instructions = "You are a helpful assistant. Answer the following questions accurately and concisely."

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    question = f"""You are grading the following question:
        {inputs['question']}.here is the real answer:
        {reference_outputs['answer']}.here is the model answer:
        {outputs['answer']}.is the model answer correct? Please respond with 'yes' or 'no'. 
        grade:
"""
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "system", "content": eval_instructions}, {"role": "user", "content": question}],
    )
    return response.choices[0].message.content

In [26]:
# Concisions
# checks wether actual output is less than 2x length of expected output
def concision(reference_outputs: dict, outputs: dict) -> bool:
    return int(len(outputs['answer'])) <= 2 * int(len(reference_outputs['answer']))


#### RUN EVALUATION

In [22]:
default_instructions = "You are a helpful assistant. Answer the following questions accurately and concisely."
def my_app(question:str, model:str="gpt-4o-mini", instructions:str=default_instructions) -> str:
    response = openai_client.chat.completions.create(
        model=model,
        messages=[{"role": "system", "content": instructions}, {"role": "user", "content": question}],
        max_tokens=100,
    )
    return response.choices[0].message.content

In [40]:
#call app for every input

def ls_target(inputs: dict) -> dict:
    return {"answer": my_app(inputs['question'])}

In [41]:

def correctness(*, inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    question = f"""
    You are grading the following question:
    {inputs['question']}

    Here is the real answer:
    {reference_outputs['answer']}

    Here is the model answer:
    {outputs['answer']}

    Is the model answer correct? Please respond with 'yes' or 'no'.
    """
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": eval_instructions},
            {"role": "user", "content": question},
        ],
    )
    return response.choices[0].message.content.strip().lower() == "yes"

def concision(*, outputs: dict, reference_outputs: dict) -> bool:
    return len(outputs["answer"]) <= 2 * len(reference_outputs["answer"])

experiment_results = client.evaluate(
    ls_target,
    data=dataset,
    evaluators=[correctness, concision],
    experiment_prefix="openai-4o-mini-chatbot",
)

View the evaluation results for experiment: 'openai-4o-mini-chatbot-7ee419f2' at:
https://smith.langchain.com/o/a891950d-da53-4f89-8994-09fbd9434adb/datasets/b8a9db35-8497-421d-9204-8093cb7b9c70/compare?selectedSessions=27721c3e-9238-409d-95c1-3bb256ade638




0it [00:00, ?it/s]

In [42]:
experiment_results = client.evaluate(
    ls_target,
    data=dataset,
    evaluators=[correctness, concision],
    experiment_prefix="openai-4-turbo",
)

View the evaluation results for experiment: 'openai-4-turbo-5bb74954' at:
https://smith.langchain.com/o/a891950d-da53-4f89-8994-09fbd9434adb/datasets/b8a9db35-8497-421d-9204-8093cb7b9c70/compare?selectedSessions=12bd6b70-ad2b-4149-97bf-09c30dad3ad3




0it [00:00, ?it/s]

## Evaluation metrics in RAG


1. Are the documents relevant?
2. Are the answers relevant?
3. Is the answer grounded in the documents retreived?
4. Does the answer match the ground truth?

Steps:

1. Create RAG: data ingestion, retrieval, generation
2. test data: question , answers
3. Evaluation metrics



In [45]:
!pip install beautifulsoup4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 5.3 MB/s eta 0:00:00

[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [46]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings

urls = [
    'https://arxiv.org/pdf/2606.03303'
]

docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [doc for sublist in docs for doc in sublist]

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=250, chunk_overlap=0)
texts = text_splitter.split_documents(docs_list)

vectorstore = InMemoryVectorStore.from_documents(texts, OpenAIEmbeddings())

retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 6})


In [47]:
retriever.invoke("what is LEAP")

[Document(id='34a8f59d-b3c2-402d-87c5-0bd501b0e125', metadata={'source': 'https://arxiv.org/pdf/2606.03303'}, page_content='LEAP: Supercharging LLMs for Formal Mathematics with Agentic FrameworksPo-Nien KungLinfeng SongDawsen HwangJinsung YoonChun-Liang LiSimone SeveriniMirek OlšákEdward LockhartQuoc V LeBurak GokturkThang LuongTomas PfisterNanyun Penghttp://creativecommons.org/licenses/by/4.0/cs.AI'),
 Document(id='816e4348-ff13-4c17-bad3-01f8805bc78b', metadata={'source': 'https://arxiv.org/pdf/2606.03303'}, page_content='<< /Author (Po-Nien Kung; Linfeng Song; Dawsen Hwang; Jinsung Yoon; Chun-Liang Li; Simone Severini; Mirek Ol\\235�k; Edward Lockhart; Quoc V Le; Burak Gokturk; Thang Luong; Tomas Pfister; Nanyun Peng) /Creator (arXiv GenPDF \\(tex2pdf:a6404ea\\)) /DOI (https://doi.org/10.48550/arXiv.2606.03303) /License (http://creativecommons.org/licenses/by/4.0/) /PTEX.Fullbanner (This is pdfTeX, Version 3.141592653-2.6-1.40.28 \\(TeX Live 2025\\) kpathsea version 6.4.1) /Producer

In [49]:
import os
from langchain.chat_models import init_chat_model
llm = init_chat_model("gpt-4o-mini", model_kwargs={"temperature": 0.0})

/var/folders/0q/vsvgsd3s4ws83m1fcrysjbvc0000gn/T/ipykernel_34420/1948774862.py:3: UserWarning: Parameters {'temperature'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  llm = init_chat_model("gpt-4o-mini", model_kwargs={"temperature": 0.0})


In [50]:
from langsmith import traceable
@traceable
def rag_bot(question: str) -> str:
    docs = retriever.invoke(question)
    context = "\n".join([doc.page_content for doc in docs])
    prompt = f"Answer the question based on the following context:\n{context}\nQuestion: {question}\nAnswer:"
    response = llm.invoke(prompt)
    return {"answer": response.content, "docs": docs}

In [51]:
rag_bot("What is LEAP?")

{'answer': 'LEAP is a framework designed to enhance large language models (LLMs) for formal mathematics by utilizing agentic frameworks. It aims to improve the capabilities of LLMs in handling mathematical reasoning and formal proofs. The work is presented by a group of authors and is available on arXiv.',
 'docs': [Document(id='34a8f59d-b3c2-402d-87c5-0bd501b0e125', metadata={'source': 'https://arxiv.org/pdf/2606.03303'}, page_content='LEAP: Supercharging LLMs for Formal Mathematics with Agentic FrameworksPo-Nien KungLinfeng SongDawsen HwangJinsung YoonChun-Liang LiSimone SeveriniMirek OlšákEdward LockhartQuoc V LeBurak GokturkThang LuongTomas PfisterNanyun Penghttp://creativecommons.org/licenses/by/4.0/cs.AI'),
  Document(id='816e4348-ff13-4c17-bad3-01f8805bc78b', metadata={'source': 'https://arxiv.org/pdf/2606.03303'}, page_content='<< /Author (Po-Nien Kung; Linfeng Song; Dawsen Hwang; Jinsung Yoon; Chun-Liang Li; Simone Severini; Mirek Ol\\235�k; Edward Lockhart; Quoc V Le; Burak G

### Dataset

In [57]:
from langsmith import Client
Client = Client()

# dataset = Client.create_dataset(
#     dataset_name="LEAP Examples"
# )

examples = [
    {
        "inputs": {"question": "What is LEAP?"},
        "outputs": {
            "answer": "LEAP is a framework for building and deploying AI models, designed to facilitate the development of machine learning applications for mathematical and formal proofs."
        },
    }
]
client.create_examples(dataset_id=dataset.id, examples=examples)

{'example_ids': ['5297933b-b28c-4821-9015-4612f5db76a0'],
 'count': 1,
 'as_of': '2026-10-03T19:46:10.980549188Z'}

In [66]:
from typing_extensions import Annotated, TypedDict

## Correctness output schema

class CorrectnessOutput(TypedDict):
    explanation: Annotated[str, "A brief explanation of why the model's answer is correct or incorrect."]
    correct: Annotated[bool, "A boolean indicating whether the model's answer is correct or not."]


#correctness prompt

correctness_prompt = """You are grading the following question:
{question}
Here is the real answer:
{reference_answer}
Here is the model answer:
{model_answer}
Please provide a brief explanation of why the model's answer is correct or incorrect, and indicate whether the model's answer is correct or not. Respond in the following JSON format:
{{
    "explanation": "<your explanation here>",
    "correct": <true or false>
}}"""

from langchain.chat_models import ChatOpenAI
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
).with_structured_output(
    schema=CorrectnessOutput,
    method="json_schema",
)

def correctness(*, inputs: dict, outputs: dict, reference_outputs: dict) -> CorrectnessOutput:
    question = correctness_prompt.format(
        question=inputs["question"],
        reference_answer=reference_outputs["answer"],
        model_answer=outputs["answer"]
    )
    response = llm.invoke(question)
    return response
